In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/city_hour.csv')

print(df.shape)
print(df.columns.tolist())
print(df.head(10))
print(df['City'].unique())

(707875, 16)
['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']
        City             Datetime  PM2.5  PM10    NO    NO2    NOx  NH3    CO  \
0  Ahmedabad  2015-01-01 01:00:00    NaN   NaN  1.00  40.01  36.37  NaN  1.00   
1  Ahmedabad  2015-01-01 02:00:00    NaN   NaN  0.02  27.75  19.73  NaN  0.02   
2  Ahmedabad  2015-01-01 03:00:00    NaN   NaN  0.08  19.32  11.08  NaN  0.08   
3  Ahmedabad  2015-01-01 04:00:00    NaN   NaN  0.30  16.45   9.20  NaN  0.30   
4  Ahmedabad  2015-01-01 05:00:00    NaN   NaN  0.12  14.90   7.85  NaN  0.12   
5  Ahmedabad  2015-01-01 06:00:00    NaN   NaN  0.33  15.95  10.82  NaN  0.33   
6  Ahmedabad  2015-01-01 07:00:00    NaN   NaN  0.45  15.94  12.47  NaN  0.45   
7  Ahmedabad  2015-01-01 08:00:00    NaN   NaN  1.03  16.66  16.48  NaN  1.03   
8  Ahmedabad  2015-01-01 09:00:00    NaN   NaN  1.47  16.25  18.02  NaN  1.47   
9  Ahmedabad  2015-01-01 10:00:00    NaN

In [2]:
delhi_df = df[df['City'] == 'Delhi'].copy()

print("Total rows:", len(delhi_df))
print("Date range:", delhi_df['Datetime'].min(), "to", delhi_df['Datetime'].max())
print("\nMissing values per column:")
print(delhi_df.isnull().sum())
print("\nRows where AQI is not null:", delhi_df['AQI'].notna().sum())

Total rows: 48192
Date range: 2015-01-01 01:00:00 to 2020-07-01 00:00:00

Missing values per column:
City              0
Datetime          0
PM2.5           375
PM10           2421
NO              298
NO2             330
NOx              25
NH3             980
CO              364
SO2            2852
O3             2201
Benzene          38
Toluene          26
Xylene        18904
AQI             498
AQI_Bucket      498
dtype: int64

Rows where AQI is not null: 47694


In [3]:
# Step 1: Filter to Delhi only and make a clean copy
delhi_df = df[df['City'] == 'Delhi'].copy()

# Step 2: Parse datetime and sort
delhi_df['Datetime'] = pd.to_datetime(delhi_df['Datetime'])
delhi_df = delhi_df.sort_values('Datetime').reset_index(drop=True)

# Step 3: Drop Xylene - too much missing data (39%)
delhi_df = delhi_df.drop(columns=['Xylene', 'AQI_Bucket'])

# Step 4: Forward fill missing values (max 3 hours gap)
# This is safe for air quality data - pollution levels don't jump instantly
pollutant_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 
                  'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene']

delhi_df[pollutant_cols] = delhi_df[pollutant_cols].ffill(limit=3)

# Step 5: For AQI specifically - drop rows where it's still missing after ffill
delhi_df = delhi_df.dropna(subset=['AQI'])

# Step 6: Verify
print("Shape after cleaning:", delhi_df.shape)
print("\nRemaining missing values:")
print(delhi_df.isnull().sum())
print("\nDate range:", delhi_df['Datetime'].min(), "to", delhi_df['Datetime'].max())

Shape after cleaning: (47694, 14)

Remaining missing values:
City           0
Datetime       0
PM2.5         56
PM10        1897
NO            31
NO2           44
NOx            4
NH3          532
CO           160
SO2         2350
O3          1755
Benzene        7
Toluene        5
AQI            0
dtype: int64

Date range: 2015-01-01 16:00:00 to 2020-07-01 00:00:00


In [4]:
# Step 7: Fill remaining gaps using median for that hour-of-day
# More intelligent than global median - respects daily pollution cycles
delhi_df['hour'] = delhi_df['Datetime'].dt.hour

for col in pollutant_cols:
    if delhi_df[col].isnull().sum() > 0:
        hourly_median = delhi_df.groupby('hour')[col].transform('median')
        delhi_df[col] = delhi_df[col].fillna(hourly_median)

# Drop the helper column
delhi_df = delhi_df.drop(columns=['hour'])

# Step 8: Final check
print("Remaining missing values after hourly median fill:")
print(delhi_df.isnull().sum())
print("\nFinal shape:", delhi_df.shape)

# Step 9: Save cleaned data
delhi_df.to_csv('../data/processed/delhi_aqi_clean.csv', index=False)
print("\nSaved to data/processed/delhi_aqi_clean.csv")

Remaining missing values after hourly median fill:
City        0
Datetime    0
PM2.5       0
PM10        0
NO          0
NO2         0
NOx         0
NH3         0
CO          0
SO2         0
O3          0
Benzene     0
Toluene     0
AQI         0
dtype: int64

Final shape: (47694, 14)

Saved to data/processed/delhi_aqi_clean.csv


In [5]:
import numpy as np

# Reload clean data fresh
delhi_df = pd.read_csv('../data/processed/delhi_aqi_clean.csv')
delhi_df['Datetime'] = pd.to_datetime(delhi_df['Datetime'])
delhi_df = delhi_df.sort_values('Datetime').reset_index(drop=True)

# ── TEMPORAL FEATURES ──────────────────────────────────────────────
delhi_df['hour']       = delhi_df['Datetime'].dt.hour
delhi_df['day_of_week']= delhi_df['Datetime'].dt.dayofweek   # 0=Monday
delhi_df['month']      = delhi_df['Datetime'].dt.month
delhi_df['is_weekend'] = (delhi_df['day_of_week'] >= 5).astype(int)
delhi_df['season']     = delhi_df['month'].map({
    12:0, 1:0, 2:0,   # Winter
    3:1,  4:1, 5:1,   # Spring
    6:2,  7:2, 8:2,   # Monsoon
    9:3, 10:3, 11:3   # Autumn
})

# ── LAG FEATURES ───────────────────────────────────────────────────
# "What was AQI N hours ago?"
for lag in [1, 2, 3, 6, 12, 24]:
    delhi_df[f'AQI_lag_{lag}h'] = delhi_df['AQI'].shift(lag)

# ── ROLLING STATISTICS ─────────────────────────────────────────────
# min_periods=1 means it won't produce NaN even at the start
delhi_df['AQI_rolling_mean_6h']  = delhi_df['AQI'].shift(1).rolling(6,  min_periods=1).mean()
delhi_df['AQI_rolling_mean_24h'] = delhi_df['AQI'].shift(1).rolling(24, min_periods=1).mean()
delhi_df['AQI_rolling_std_24h']  = delhi_df['AQI'].shift(1).rolling(24, min_periods=1).std()
delhi_df['AQI_rolling_max_24h']  = delhi_df['AQI'].shift(1).rolling(24, min_periods=1).max()

# ── DROP ROWS WITH NaN from lagging ────────────────────────────────
# First 24 rows will have NaN lag values - drop them
delhi_df = delhi_df.dropna(subset=['AQI_lag_24h']).reset_index(drop=True)

# ── VERIFY ─────────────────────────────────────────────────────────
print("Shape after feature engineering:", delhi_df.shape)
print("\nNew features added:")
new_cols = [c for c in delhi_df.columns if c not in 
            ['City','Datetime','PM2.5','PM10','NO','NO2','NOx',
             'NH3','CO','SO2','O3','Benzene','Toluene','AQI']]
print(new_cols)
print("\nSample row to verify no NaNs in new features:")
print(delhi_df[new_cols].isnull().sum())

Shape after feature engineering: (47670, 29)

New features added:
['hour', 'day_of_week', 'month', 'is_weekend', 'season', 'AQI_lag_1h', 'AQI_lag_2h', 'AQI_lag_3h', 'AQI_lag_6h', 'AQI_lag_12h', 'AQI_lag_24h', 'AQI_rolling_mean_6h', 'AQI_rolling_mean_24h', 'AQI_rolling_std_24h', 'AQI_rolling_max_24h']

Sample row to verify no NaNs in new features:
hour                    0
day_of_week             0
month                   0
is_weekend              0
season                  0
AQI_lag_1h              0
AQI_lag_2h              0
AQI_lag_3h              0
AQI_lag_6h              0
AQI_lag_12h             0
AQI_lag_24h             0
AQI_rolling_mean_6h     0
AQI_rolling_mean_24h    0
AQI_rolling_std_24h     0
AQI_rolling_max_24h     0
dtype: int64


In [6]:
# Save featured dataset
delhi_df.to_parquet('../data/processed/delhi_aqi_features.parquet', index=False)
print("Saved to data/processed/delhi_aqi_features.parquet")
print("Shape:", delhi_df.shape)
print("Columns:", delhi_df.columns.tolist())

Saved to data/processed/delhi_aqi_features.parquet
Shape: (47670, 29)
Columns: ['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'AQI', 'hour', 'day_of_week', 'month', 'is_weekend', 'season', 'AQI_lag_1h', 'AQI_lag_2h', 'AQI_lag_3h', 'AQI_lag_6h', 'AQI_lag_12h', 'AQI_lag_24h', 'AQI_rolling_mean_6h', 'AQI_rolling_mean_24h', 'AQI_rolling_std_24h', 'AQI_rolling_max_24h']


In [7]:
# Reload clean data
delhi_df = pd.read_csv('../data/processed/delhi_aqi_clean.csv')
delhi_df['Datetime'] = pd.to_datetime(delhi_df['Datetime'])
delhi_df = delhi_df.sort_values('Datetime').reset_index(drop=True)

# ── TARGET: AQI 24 hours from now ─────────────────────────────────
# shift(-24) means "what will AQI be 24 hours later"
delhi_df['AQI_target_24h'] = delhi_df['AQI'].shift(-24)

# ── TEMPORAL FEATURES ─────────────────────────────────────────────
delhi_df['hour']        = delhi_df['Datetime'].dt.hour
delhi_df['day_of_week'] = delhi_df['Datetime'].dt.dayofweek
delhi_df['month']       = delhi_df['Datetime'].dt.month
delhi_df['is_weekend']  = (delhi_df['day_of_week'] >= 5).astype(int)
delhi_df['season']      = delhi_df['month'].map({
    12:0, 1:0, 2:0,
    3:1,  4:1, 5:1,
    6:2,  7:2, 8:2,
    9:3, 10:3, 11:3
})

# ── LAG FEATURES ──────────────────────────────────────────────────
# Minimum safe lag is 24h since we're predicting 24h ahead
# Using 24h, 48h, 72h — data you'd actually HAVE at prediction time
for lag in [24, 48, 72]:
    delhi_df[f'AQI_lag_{lag}h'] = delhi_df['AQI'].shift(lag)

# Same day last week
delhi_df['AQI_lag_168h'] = delhi_df['AQI'].shift(168)

# ── ROLLING STATISTICS ────────────────────────────────────────────
# All rolled from 24h+ back — no leakage
delhi_df['AQI_rolling_mean_24h'] = delhi_df['AQI'].shift(24).rolling(24, min_periods=1).mean()
delhi_df['AQI_rolling_mean_48h'] = delhi_df['AQI'].shift(24).rolling(48, min_periods=1).mean()
delhi_df['AQI_rolling_std_24h']  = delhi_df['AQI'].shift(24).rolling(24, min_periods=1).std()
delhi_df['AQI_rolling_max_24h']  = delhi_df['AQI'].shift(24).rolling(24, min_periods=1).max()
delhi_df['AQI_rolling_min_24h']  = delhi_df['AQI'].shift(24).rolling(24, min_periods=1).min()

# ── DROP ROWS WITH NaN ────────────────────────────────────────────
# Lose first 168 rows (lag_168h) and last 24 rows (target shift)
delhi_df = delhi_df.dropna(subset=['AQI_lag_168h', 'AQI_target_24h']).reset_index(drop=True)

# ── VERIFY ────────────────────────────────────────────────────────
print("Shape:", delhi_df.shape)
print("\nMissing values:")
print(delhi_df.isnull().sum())

new_features = [
    'hour', 'day_of_week', 'month', 'is_weekend', 'season',
    'AQI_lag_24h', 'AQI_lag_48h', 'AQI_lag_72h', 'AQI_lag_168h',
    'AQI_rolling_mean_24h', 'AQI_rolling_mean_48h',
    'AQI_rolling_std_24h', 'AQI_rolling_max_24h', 'AQI_rolling_min_24h'
]
print("\nFeature count:", len(new_features))

Shape: (47502, 29)

Missing values:
City                    0
Datetime                0
PM2.5                   0
PM10                    0
NO                      0
NO2                     0
NOx                     0
NH3                     0
CO                      0
SO2                     0
O3                      0
Benzene                 0
Toluene                 0
AQI                     0
AQI_target_24h          0
hour                    0
day_of_week             0
month                   0
is_weekend              0
season                  0
AQI_lag_24h             0
AQI_lag_48h             0
AQI_lag_72h             0
AQI_lag_168h            0
AQI_rolling_mean_24h    0
AQI_rolling_mean_48h    0
AQI_rolling_std_24h     0
AQI_rolling_max_24h     0
AQI_rolling_min_24h     0
dtype: int64

Feature count: 14


In [8]:
# Save the 24h forecast dataset
delhi_df.to_parquet('../data/processed/delhi_aqi_24h_features.parquet', index=False)
print("Saved to data/processed/delhi_aqi_24h_features.parquet")

Saved to data/processed/delhi_aqi_24h_features.parquet
